## All Techniques of Hyper Parameter Optimization
- GridSearchCV
- RandomizedSearchCV
- Bayesian Optimization - Automate Hyperparamtere Tuning(Hyperopt)
- Sequential Model Based Optimization (Tuninga scikit-learn estimator with skopt)
- Optuna-Automate Hyperparameter Tuning
- Genetic Algorithms (TPOT Classifier)

### Refrences
- https://github.com/fmfn/BayesianOptimization
- https://github.com/hyperopt/hyperopt
- https://www.jeremyjordan.me/hyperparameter-tuning/
- https://optuna.org/
- https://towardsdatascience.com/hyperparameters-optimization-526348bb8e2d
- http://scikit-optimize.github.io/stable/auto_examples/hyperparameter-optimization.html

In [14]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
from pathlib import Path

## Read dataset
DATA_DIR = Path.cwd().parent / "data" / "advandced_hyperparameter_tunning"
df = pd.read_csv(DATA_DIR / "diabetes.csv")

In [25]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [26]:
import numpy as np

## converting glucose and Insulin features into one-hot-encoding
df['Glucose'] = np.where(df.Glucose == 0, df.Glucose.median(), df['Glucose'])
df['Insulin'] = np.where(df.Insulin == 0, df.Insulin.median(), df['Insulin'])

In [27]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72,35,30.5,33.6,0.627,50,1
1,1,85.0,66,29,30.5,26.6,0.351,31,0
2,8,183.0,64,0,30.5,23.3,0.672,32,1
3,1,89.0,66,23,94.0,28.1,0.167,21,0
4,0,137.0,40,35,168.0,43.1,2.288,33,1


In [28]:
## Independent And Dependent Features

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [29]:
X.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148.0,72,35,30.5,33.6,0.627,50
1,1,85.0,66,29,30.5,26.6,0.351,31
2,8,183.0,64,0,30.5,23.3,0.672,32
3,1,89.0,66,23,94.0,28.1,0.167,21
4,0,137.0,40,35,168.0,43.1,2.288,33


In [30]:
## Labels

y.head()

0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

In [32]:
## train test split

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [81]:
from hyperopt import hp, fmin, tpe, STATUS_OK, Trials

---## Standardized ML Pipeline*Auto-generated by Phase 5 ML Standardization***STEP 1** — LazyPredict baseline comparison  **STEP 2** — PyCaret automated pipeline

### STEP 1 — LazyPredict: Baseline Model ComparisonRun all sklearn-compatible models to find the best baseline.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from lazypredict.Supervised import LazyClassifier

# Use existing train/test split from preprocessing above
# Ensure numeric-only for LazyPredict (handles mixed types)
try:
    X_train_lp = X_train.select_dtypes(include=['number']).fillna(0)
    X_test_lp = X_test.select_dtypes(include=['number']).fillna(0)
except AttributeError:
    X_train_lp, X_test_lp = X_train, X_test

# Run LazyPredict
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models_df, predictions_df = clf.fit(X_train_lp, X_test_lp, y_train, y_test)

print("=" * 60)
print("LazyPredict — Classification Baseline Results")
print("=" * 60)
models_df

#### Top Models Visualization

In [ ]:
import matplotlib.pyplot as plt

top_n = min(15, len(models_df))
fig, ax = plt.subplots(figsize=(10, 6))
models_df['Accuracy'].head(top_n).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Accuracy')
ax.set_title(f'Top {top_n} Models — Accuracy')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### STEP 2 — PyCaret: Automated ML PipelineFull pipeline with automated preprocessing, model comparison, tuning, and finalization.> **Note:** PyCaret requires Python 3.9–3.11. Install with: `pip install pycaret`

In [ ]:
import sys
import pandas as pd

# PyCaret setup
try:
    from pycaret.classification import setup, compare_models, finalize_model, predict_model, save_model, plot_model
except ImportError:
    print("PyCaret not installed. Install with: pip install pycaret")
    print("Requires Python 3.9-3.11")
    raise SystemExit("PyCaret required for STEP 2")

# Safety check — target variable must exist
assert y_train is not None, "y_train is None — cannot proceed"

# Reconstruct DataFrame for PyCaret (needs column-named target)
# IMPORTANT: Do NOT mutate original X_train / y_train
if isinstance(X_train, pd.DataFrame):
    df_train = X_train.copy()
else:
    df_train = pd.DataFrame(X_train)
df_train['__target__'] = y_train

if isinstance(X_test, pd.DataFrame):
    df_test = X_test.copy()
else:
    df_test = pd.DataFrame(X_test)
df_test['__target__'] = y_test

_full_df = pd.concat([df_train, df_test], ignore_index=True)

# Configure PyCaret session
clf_setup = setup(
    data=_full_df,
    target='__target__',
    session_id=42,
    silent=True,
    verbose=False,
    fold=5,
)
print("PyCaret setup complete.")

In [ ]:
# Compare all models
best_model = compare_models(sort='Accuracy', n_select=1)
print(f"\nBest model: {best_model}")

In [ ]:
# Finalize the best model (retrain on full dataset)
final_model = finalize_model(best_model)
print(f"Finalized model: {final_model}")

#### Model Evaluation

In [ ]:
# Model evaluation plots
try:
    plot_model(best_model, plot='confusion_matrix', save=True)
    plot_model(best_model, plot='auc', save=True)
    plot_model(best_model, plot='feature', save=True)
except Exception as e:
    print(f"Plot generation note: {e}")

#### Save Model Pipeline

In [ ]:
# Save the final pipeline
save_model(final_model, 'final_model_pipeline')
print("Model saved as 'final_model_pipeline.pkl'")